# PaddleOCR-VL-1.6 — V6 isolated dependency builder

Run this notebook as a Kaggle **Save Version** with **Internet ON** and **Accelerator: None**. It creates one attachable output bundle containing:

- the pinned PaddleOCR client and vLLM/LLM-Compressor wheelhouses;
- the BF16 PaddleOCR-VL-1.6 source model and frozen PP-DocLayoutV3 model;
- the complete 1,651-page OmniDocBench test set;
- a separate 512-page, category-balanced DocLayNet calibration set;
- the offline GPTQ worker that the GPU notebook will use to create **W8A16-G128** and **W4A16-G128**.

This notebook never imports CUDA, launches quantization, or requires a GPU. Every candidate calibration page is screened against all 1,651 OmniDocBench pages using exact, normalized-pixel, and perceptual fingerprints. Any overlap, missing wheel, corrupt wheel, or source-contract drift stops the notebook. Quantization and benchmarking happen later with Internet OFF in notebook 2.


**File identity:** `01_DEPENDENCY_FIXED_V6_NO_RESOLVER_CONFLICT.ipynb`  
**Resolver correction:** vLLM inference and LLM Compressor quantization are downloaded independently and are never passed to one pip resolver call.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib, io, json, os, platform, random, re, shutil, struct, subprocess, sys, time, zipfile

ROOT = Path('/kaggle/working/paddleocr_vl_1_6_l4_wna16_bundle')
WHEELS = ROOT / 'wheels'
CLIENT_WHEELS = WHEELS / 'paddle_client'
SERVER_WHEELS = WHEELS / 'vllm_server'
QUANT_WHEELS = WHEELS / 'llmcompressor_quant'
MODELS = ROOT / 'models'
BF16 = MODELS / 'PaddleOCR-VL-1.6-BF16'
LAYOUT = MODELS / 'PP-DocLayoutV3'
DATASETS = ROOT / 'datasets'
OMNI = DATASETS / 'OmniDocBench'
CALIBRATION = DATASETS / 'calibration_doclaynet'
CALIBRATION_DOWNLOAD = Path('/kaggle/working/l4_wna16_doclaynet_download_temp')
SCRIPTS = ROOT / 'scripts'
LOGS = ROOT / 'logs'
EXPECTED_PAGES = 1651
EXPECTED_CALIBRATION = 512
SEED = 2026

VERSIONS = {
    'python': '3.12',
    'paddlepaddle-gpu': '3.2.1',
    'paddleocr': '3.7.0',
    'paddlex': '3.7.2',
    'vllm': '0.29.0',
    'llmcompressor': '0.12.0.1',
}
REVISIONS = {
    'PaddlePaddle/PaddleOCR-VL-1.6': 'c5630abae1d940eafe0697512a0325494b02ab42',
    'PaddlePaddle/PP-DocLayoutV3': '7b48a7566925fa464281f930c58eee04fe2c862a',
    'opendatalab/OmniDocBench': 'aa1ee96d106dbe53d0ae59474d75c6e6d9b53fec',
    # The exact calibration revision is resolved and frozen later in this notebook.
    'pierreguillou/DocLayNet-base': None,
}

if ROOT.exists():
    shutil.rmtree(ROOT)
shutil.rmtree(CALIBRATION_DOWNLOAD, ignore_errors=True)
CALIBRATION_DOWNLOAD.mkdir(parents=True)
for path in (CLIENT_WHEELS, SERVER_WHEELS, QUANT_WHEELS, MODELS, DATASETS, CALIBRATION, SCRIPTS, LOGS):
    path.mkdir(parents=True, exist_ok=True)

def run_checked(command, **kwargs):
    print('+', ' '.join(map(str, command)), flush=True)
    return subprocess.run(command, check=True, **kwargs)

print('Python:', sys.version)
print('Platform:', platform.platform())
assert sys.version_info[:2] == (3, 12), sys.version
print('Output bundle:', ROOT)
print('Builder mode: Internet ON, Accelerator None (no CUDA work in this notebook)')


## 1. Build and integrity-check three isolated wheelhouses

Paddle uses the official stable CUDA 12.6 index. vLLM inference and LLM Compressor quantization are resolved into separate wheelhouses because they intentionally use different, internally consistent dependency stacks. Downloads go into a temporary attempt directory, every wheel ZIP is tested, and only a complete attempt is moved into the bundle.


In [ ]:
os.environ['PIP_NO_CACHE_DIR'] = '1'
PADDLE_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PYPI_INDEX = 'https://pypi.org/simple'

def verify_wheels(directory, require_prefixes=()):
    wheels = sorted(Path(directory).glob('*.whl'))
    if not wheels:
        raise RuntimeError(f'No wheels downloaded into {directory}')
    for wheel in wheels:
        if wheel.stat().st_size == 0:
            raise RuntimeError(f'Empty wheel: {wheel}')
        try:
            with zipfile.ZipFile(wheel) as archive:
                bad_member = archive.testzip()
                if bad_member:
                    raise RuntimeError(f'Corrupt member {bad_member} in {wheel}')
        except zipfile.BadZipFile as exc:
            raise RuntimeError(f'Invalid wheel archive: {wheel}') from exc
    names = [wheel.name.lower().replace('_', '-') for wheel in wheels]
    for prefix in require_prefixes:
        canonical = prefix.lower().replace('_', '-')
        if not any(name.startswith(canonical) for name in names):
            raise RuntimeError(f'Missing required wheel prefix {prefix} in {directory}')
    return wheels

def robust_pip_download(label, packages, destination, extra_args=(), require_prefixes=()):
    package_names = {
        package.split('==', 1)[0].split('[', 1)[0].lower().replace('_', '-')
        for package in packages
    }
    if {'vllm', 'llmcompressor'} <= package_names:
        raise RuntimeError(
            'Invalid combined resolver request: vLLM and LLM Compressor must use separate wheelhouses.'
        )
    destination = Path(destination)
    last_error = None
    for attempt in range(1, 4):
        attempt_dir = Path('/tmp') / f'l4_wna16_{label}_attempt_{attempt}'
        shutil.rmtree(attempt_dir, ignore_errors=True)
        attempt_dir.mkdir(parents=True)
        command = [
            sys.executable, '-m', 'pip', 'download', '--only-binary=:all:',
            '--dest', str(attempt_dir), *extra_args, *packages,
        ]
        try:
            run_checked(command)
            wheels = verify_wheels(attempt_dir, require_prefixes)
            for wheel in wheels:
                target = destination / wheel.name
                if target.exists():
                    target.unlink()
                shutil.move(str(wheel), str(target))
            shutil.rmtree(attempt_dir, ignore_errors=True)
            return
        except Exception as exc:
            last_error = exc
            print(f'{label} attempt {attempt}/3 failed: {exc}', flush=True)
            shutil.rmtree(attempt_dir, ignore_errors=True)
            if attempt < 3:
                time.sleep(5 * attempt)
    raise RuntimeError(f'{label} failed after three complete attempts') from last_error

robust_pip_download(
    'paddle_runtime',
    [f"paddlepaddle-gpu=={VERSIONS['paddlepaddle-gpu']}", 'numpy==2.2.6'],
    CLIENT_WHEELS,
    extra_args=('-i', PADDLE_INDEX, '--extra-index-url', PYPI_INDEX),
    require_prefixes=(f"paddlepaddle_gpu-{VERSIONS['paddlepaddle-gpu']}", 'numpy-2.2.6'),
)
robust_pip_download(
    'paddle_client',
    [
        f"paddleocr[doc-parser]=={VERSIONS['paddleocr']}",
        f"paddlex[ocr-core]=={VERSIONS['paddlex']}",
        'numpy==2.2.6', 'safetensors==0.7.0', 'huggingface-hub==1.11.0',
    ],
    CLIENT_WHEELS,
    extra_args=('-i', PYPI_INDEX),
    require_prefixes=(f"paddleocr-{VERSIONS['paddleocr']}", f"paddlex-{VERSIONS['paddlex']}"),
)
robust_pip_download(
    'vllm_server',
    [f"vllm=={VERSIONS['vllm']}"],
    SERVER_WHEELS,
    extra_args=('--pre', '-i', PYPI_INDEX),
    require_prefixes=(f"vllm-{VERSIONS['vllm']}", 'compressed_tensors-0.17.0'),
)
robust_pip_download(
    'llmcompressor_quant',
    [f"llmcompressor=={VERSIONS['llmcompressor']}"],
    QUANT_WHEELS,
    extra_args=('--pre', '-i', PYPI_INDEX),
    require_prefixes=(f"llmcompressor-{VERSIONS['llmcompressor']}", 'compressed_tensors-0.17.1'),
)

client_wheels = verify_wheels(CLIENT_WHEELS, ('paddlepaddle_gpu-3.2.1', 'paddleocr-3.7.0', 'paddlex-3.7.2'))
server_wheels = verify_wheels(SERVER_WHEELS, ('vllm-0.29.0', 'compressed_tensors-0.17.0'))
quant_wheels = verify_wheels(QUANT_WHEELS, ('llmcompressor-0.12.0.1', 'compressed_tensors-0.17.1'))
print('Client wheels:', len(client_wheels), f'{sum(p.stat().st_size for p in client_wheels)/2**30:.3f} GiB')
print('vLLM server wheels:', len(server_wheels), f'{sum(p.stat().st_size for p in server_wheels)/2**30:.3f} GiB')
print('LLM Compressor wheels:', len(quant_wheels), f'{sum(p.stat().st_size for p in quant_wheels)/2**30:.3f} GiB')


## 2. Download pinned source assets and an independent calibration set

OmniDocBench remains test-only. Calibration uses 512 deterministically selected pages from the randomized DocLayNet-base training pool (6,910 pages), balanced across its six document categories and restricted to one page per source document. The pinned raw ZIP is read directly for metadata and images; the obsolete dataset script's incorrect integer bbox schema is never materialized. The accepted images are stored in the output bundle so W8A16 and W4A16 receive byte-identical inputs.


In [ ]:
TOOLS_SITE = Path('/tmp/l4_wna16_builder_tools_site')
shutil.rmtree(TOOLS_SITE, ignore_errors=True)
run_checked([
    sys.executable, '-m', 'pip', 'install', '-q', '--target', str(TOOLS_SITE),
    'huggingface-hub==1.11.0', 'ImageHash==4.3.2', 'Pillow', 'tqdm',
])
sys.path.insert(0, str(TOOLS_SITE))
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
import imagehash
from tqdm.auto import tqdm

os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '90'

snapshot_download(
    repo_id='PaddlePaddle/PaddleOCR-VL-1.6', repo_type='model',
    revision=REVISIONS['PaddlePaddle/PaddleOCR-VL-1.6'], local_dir=str(BF16), max_workers=4,
)
snapshot_download(
    repo_id='PaddlePaddle/PP-DocLayoutV3', repo_type='model',
    revision=REVISIONS['PaddlePaddle/PP-DocLayoutV3'], local_dir=str(LAYOUT), max_workers=4,
)
snapshot_download(
    repo_id='opendatalab/OmniDocBench', repo_type='dataset',
    revision=REVISIONS['opendatalab/OmniDocBench'], local_dir=str(OMNI), max_workers=4,
)

calibration_repo = 'pierreguillou/DocLayNet-base'
calibration_revision = HfApi().dataset_info(calibration_repo).sha
if not calibration_revision:
    raise RuntimeError('Could not resolve the calibration dataset commit.')
REVISIONS[calibration_repo] = calibration_revision
calibration_archive = Path(hf_hub_download(
    repo_id=calibration_repo, repo_type='dataset', filename='data/dataset_base.zip',
    revision=calibration_revision, local_dir=str(CALIBRATION_DOWNLOAD),
))
if not calibration_archive.is_file() or calibration_archive.stat().st_size == 0:
    raise RuntimeError(f'DocLayNet archive is missing or empty: {calibration_archive}')
(ROOT / 'source_revisions.json').write_text(
    json.dumps(REVISIONS, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)

for cache_dir in sorted(ROOT.rglob('.cache'), reverse=True):
    if cache_dir.is_dir():
        shutil.rmtree(cache_dir, ignore_errors=True)

print('BF16 model:', BF16)
print('Layout model:', LAYOUT)
print('OmniDocBench:', OMNI)
print('DocLayNet-base calibration revision:', calibration_revision)
print('DocLayNet raw archive:', calibration_archive)


## 3. Validate all inputs and freeze the experimental contract


In [ ]:
def require_file(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Required file missing or empty: {path}')
    return path

for filename in [
    'config.json', 'generation_config.json', 'model.safetensors',
    'inference.yml', 'preprocessor_config.json', 'tokenizer.json',
    'tokenizer_config.json', 'modeling_paddleocr_vl.py',
    'configuration_paddleocr_vl.py', 'processing_paddleocr_vl.py',
    'image_processing_paddleocr_vl.py',
]:
    require_file(BF16 / filename)

with (BF16 / 'model.safetensors').open('rb') as stream:
    header_size_raw = stream.read(8)
    if len(header_size_raw) != 8:
        raise RuntimeError('Invalid BF16 safetensors header.')
    header_size = struct.unpack('<Q', header_size_raw)[0]
    if not 0 < header_size < min(100_000_000, (BF16 / 'model.safetensors').stat().st_size - 8):
        raise RuntimeError('Invalid BF16 safetensors header size.')
    tensor_header = json.loads(stream.read(header_size))
source_dtypes = sorted({item['dtype'] for name, item in tensor_header.items() if name != '__metadata__'})
assert source_dtypes == ['BF16'], source_dtypes

for filename in ['inference.yml', 'inference.pdiparams', 'inference.json']:
    require_file(LAYOUT / filename)

annotations = json.loads(require_file(OMNI / 'OmniDocBench.json').read_text(encoding='utf-8'))
assert isinstance(annotations, list) and len(annotations) == EXPECTED_PAGES
images_dir = OMNI / 'images'
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
images_by_name = defaultdict(list)
for path in images_dir.rglob('*'):
    if path.is_file() and path.suffix.lower() in image_extensions:
        images_by_name[path.name].append(path)

benchmark_inputs = []
seen_page_ids = set()
for index, annotation in enumerate(tqdm(annotations, desc='Validating OmniDocBench pages')):
    image_reference = annotation.get('page_info', {}).get('image_path')
    if not isinstance(image_reference, str) or not image_reference:
        raise RuntimeError(f'Missing image_path in annotation {index}.')
    filename = image_reference.replace('\\', '/').rsplit('/', 1)[-1]
    candidates = images_by_name.get(filename, [])
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one image for {filename}; found {len(candidates)}.')
    image_path = require_file(candidates[0])
    page_id = image_path.stem
    if page_id in seen_page_ids:
        raise RuntimeError(f'Duplicate page ID: {page_id}')
    seen_page_ids.add(page_id)
    with Image.open(image_path) as image:
        image.verify()
    benchmark_inputs.append({
        'annotation_index': index, 'page_id': page_id,
        'image': image_path.relative_to(ROOT).as_posix(),
    })

(ROOT / 'benchmark_inputs.json').write_text(
    json.dumps(benchmark_inputs, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)

# Build image-level fingerprints for every protected OmniDocBench test page.
# Near-duplicates are screened with a 256-bit pHash plus a conservative
# conjunction of 256-bit dHash and aHash. Exact encoded-file and normalized-
# pixel identities are checked independently.
LANCZOS = getattr(Image, 'Resampling', Image).LANCZOS
DHASH_THRESHOLD = 12
AHASH_THRESHOLD = 20
PHASH_THRESHOLD = 18

def fingerprint_image(image):
    rgb = image.convert('RGB')
    normalized = rgb.resize((256, 256), LANCZOS)
    normalized_sha256 = hashlib.sha256(normalized.tobytes()).hexdigest()

    d_image = rgb.convert('L').resize((17, 16), LANCZOS)
    d_pixels = list(d_image.getdata())
    d_value = 0
    for y in range(16):
        for x in range(16):
            d_value = (d_value << 1) | int(d_pixels[y * 17 + x] > d_pixels[y * 17 + x + 1])

    a_image = rgb.convert('L').resize((16, 16), LANCZOS)
    a_pixels = list(a_image.getdata())
    average = sum(a_pixels) / len(a_pixels)
    a_value = 0
    for value in a_pixels:
        a_value = (a_value << 1) | int(value >= average)
    return {
        'normalized_rgb_256_sha256': normalized_sha256,
        'phash256': str(imagehash.phash(rgb, hash_size=16, highfreq_factor=4)),
        'dhash256': f'{d_value:064x}',
        'ahash256': f'{a_value:064x}',
    }

def hamming_hex(left, right):
    return (int(left, 16) ^ int(right, 16)).bit_count()

def perceptually_close(left, right):
    return (
        hamming_hex(left['phash256'], right['phash256']) <= PHASH_THRESHOLD
        or (
            hamming_hex(left['dhash256'], right['dhash256']) <= DHASH_THRESHOLD
            and hamming_hex(left['ahash256'], right['ahash256']) <= AHASH_THRESHOLD
        )
    )

omni_fingerprints = []
omni_file_hashes = set()
omni_normalized_hashes = set()
for entry in tqdm(benchmark_inputs, desc='Fingerprinting protected OmniDocBench pages'):
    path = ROOT / entry['image']
    file_sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
    with Image.open(path) as opened:
        fingerprints = fingerprint_image(opened)
    omni_file_hashes.add(file_sha256)
    omni_normalized_hashes.add(fingerprints['normalized_rgb_256_sha256'])
    omni_fingerprints.append({
        'page_id': entry['page_id'], 'file_sha256': file_sha256, **fingerprints,
    })
(ROOT / 'omnidocbench_image_fingerprints.json').write_text(
    json.dumps(omni_fingerprints, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)

# Draw a balanced, document-diverse calibration set from DocLayNet train only.
categories = [
    'financial_reports', 'scientific_articles', 'laws_and_regulations',
    'government_tenders', 'manuals', 'patents',
]
category_quota = {category: EXPECTED_CALIBRATION // len(categories) for category in categories}
for category in categories[:EXPECTED_CALIBRATION % len(categories)]:
    category_quota[category] += 1

def iter_doclaynet_train_samples(archive_path):
    # Read only metadata/images from the pinned ZIP; bypass its broken Arrow bbox schema.
    with zipfile.ZipFile(archive_path) as archive:
        members = set(archive.namelist())
        annotation_members = sorted(
            name for name in members
            if name.startswith('base_dataset/train/annotations/') and name.endswith('.json')
        )
        if len(annotation_members) != 6910:
            raise RuntimeError(
                f'Expected 6,910 DocLayNet train annotations in the pinned archive, '
                f'found {len(annotation_members)}.'
            )
        random.Random(SEED).shuffle(annotation_members)
        for annotation_member in annotation_members:
            raw = archive.read(annotation_member)
            data = json.loads(raw.decode('utf-8'))
            metadata = data.get('metadata')
            if not isinstance(metadata, dict):
                raise RuntimeError(f'Missing metadata in {annotation_member}')
            image_member = annotation_member.replace('/annotations/', '/images/')
            image_member = image_member.rsplit('.', 1)[0] + '.png'
            if image_member not in members:
                raise RuntimeError(f'Missing paired image for {annotation_member}: {image_member}')
            yield {
                'id': Path(annotation_member).stem,
                'metadata': metadata,
                '_image_member': image_member,
            }, archive

calibration_records = []
selected_documents = set()
selected_normalized_hashes = set()
selected_fingerprints = []
selected_per_category = defaultdict(int)
rejections = defaultdict(int)
candidate_count = 0

for sample, doclaynet_archive in iter_doclaynet_train_samples(calibration_archive):
    candidate_count += 1
    if candidate_count > 50_000:
        break
    metadata = sample.get('metadata') or {}
    category = metadata.get('doc_category') or sample.get('doc_category')
    if category not in category_quota or selected_per_category[category] >= category_quota[category]:
        rejections['category_full_or_unknown'] += 1
        continue
    if int(metadata.get('precedence') or 0) != 0:
        rejections['redundant_annotation'] += 1
        continue

    document_name = (
        metadata.get('original_filename') or metadata.get('doc_name')
        or sample.get('original_filename') or sample.get('doc_name')
        or metadata.get('page_hash') or sample.get('page_hash')
        or str(sample.get('image_id') or sample.get('id') or candidate_count)
    )
    document_key = (metadata.get('collection') or sample.get('collection'), document_name)
    if document_key in selected_documents:
        rejections['same_source_document'] += 1
        continue

    try:
        image_bytes = doclaynet_archive.read(sample['_image_member'])
        with Image.open(io.BytesIO(image_bytes)) as opened:
            image = opened.convert('RGB')
            image.load()
    except Exception as exc:
        raise RuntimeError(f'Could not decode {sample["_image_member"]}') from exc
    fingerprints = fingerprint_image(image)
    if fingerprints['normalized_rgb_256_sha256'] in omni_normalized_hashes:
        rejections['omnidocbench_normalized_exact'] += 1
        continue
    if any(perceptually_close(fingerprints, protected) for protected in omni_fingerprints):
        rejections['omnidocbench_perceptual_near_duplicate'] += 1
        continue
    if fingerprints['normalized_rgb_256_sha256'] in selected_normalized_hashes:
        rejections['calibration_exact_duplicate'] += 1
        continue
    if any(perceptually_close(fingerprints, prior) for prior in selected_fingerprints):
        rejections['calibration_perceptual_near_duplicate'] += 1
        continue

    output_index = len(calibration_records)
    image_path = CALIBRATION / f'doclaynet_{output_index:04d}.png'
    image.save(image_path, format='PNG', optimize=False)
    with Image.open(image_path) as check:
        check.verify()
    file_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
    if file_sha256 in omni_file_hashes:
        raise RuntimeError(f'Exact encoded-image overlap with OmniDocBench: {image_path}')

    calibration_records.append({
        'index': output_index,
        'image': image_path.relative_to(ROOT).as_posix(),
        'sha256': file_sha256,
        'source_dataset': calibration_repo,
        'source_revision': calibration_revision,
        'source_split': 'train',
        'source_image_id': sample.get('image_id') or sample.get('id') or metadata.get('image_id'),
        'source_document': document_name,
        'source_page_number': metadata.get('page_no') if metadata.get('page_no') is not None else sample.get('page_no'),
        'source_page_hash': metadata.get('page_hash') or sample.get('page_hash'),
        'doc_category': category,
        **fingerprints,
    })
    selected_documents.add(document_key)
    selected_normalized_hashes.add(fingerprints['normalized_rgb_256_sha256'])
    selected_fingerprints.append(fingerprints)
    selected_per_category[category] += 1
    if len(calibration_records) == EXPECTED_CALIBRATION:
        break

if len(calibration_records) != EXPECTED_CALIBRATION:
    raise RuntimeError(
        f'Could select only {len(calibration_records)}/{EXPECTED_CALIBRATION} non-overlapping DocLayNet pages; '
        f'category counts={dict(selected_per_category)}, rejections={dict(rejections)}'
    )
assert dict(selected_per_category) == category_quota
assert len(selected_documents) == EXPECTED_CALIBRATION

# Re-run the overlap screen from the saved PNG files. The audit is a hard gate.
minimum_dhash_distance = 256
minimum_ahash_distance = 256
minimum_phash_distance = 256
for record in tqdm(calibration_records, desc='Auditing calibration/test separation'):
    path = ROOT / record['image']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == record['sha256']
    with Image.open(path) as opened:
        saved_fingerprints = fingerprint_image(opened)
    assert saved_fingerprints['normalized_rgb_256_sha256'] == record['normalized_rgb_256_sha256']
    assert record['sha256'] not in omni_file_hashes
    assert record['normalized_rgb_256_sha256'] not in omni_normalized_hashes
    for protected in omni_fingerprints:
        p_distance = hamming_hex(record['phash256'], protected['phash256'])
        d_distance = hamming_hex(record['dhash256'], protected['dhash256'])
        a_distance = hamming_hex(record['ahash256'], protected['ahash256'])
        minimum_phash_distance = min(minimum_phash_distance, p_distance)
        minimum_dhash_distance = min(minimum_dhash_distance, d_distance)
        minimum_ahash_distance = min(minimum_ahash_distance, a_distance)
        if p_distance <= PHASH_THRESHOLD or (d_distance <= DHASH_THRESHOLD and a_distance <= AHASH_THRESHOLD):
            raise RuntimeError(
                f'Perceptual calibration/test overlap: {record["image"]} vs {protected["page_id"]} '
                f'(pHash={p_distance}, dHash={d_distance}, aHash={a_distance})'
            )

overlap_audit = {
    'status': 'PASS',
    'calibration_dataset': calibration_repo,
    'calibration_revision': calibration_revision,
    'calibration_split': 'train',
    'ingestion': 'direct pinned ZIP metadata/image reader; annotation bboxes intentionally not materialized',
    'calibration_images': len(calibration_records),
    'calibration_unique_source_documents': len(selected_documents),
    'omnidocbench_images_compared': len(omni_fingerprints),
    'candidate_images_scanned': candidate_count,
    'category_quota': category_quota,
    'selected_per_category': dict(selected_per_category),
    'rejections': dict(rejections),
    'tests': {
        'encoded_file_sha256': 'no matches',
        'normalized_rgb_256_sha256': 'no matches',
        'phash256_threshold': PHASH_THRESHOLD,
        'dhash256_threshold': DHASH_THRESHOLD,
        'ahash256_threshold': AHASH_THRESHOLD,
        'near_duplicate_rule': 'reject on pHash threshold, or when both dHash and aHash meet their thresholds',
    },
    'minimum_observed_phash_distance': minimum_phash_distance,
    'minimum_observed_dhash_distance': minimum_dhash_distance,
    'minimum_observed_ahash_distance': minimum_ahash_distance,
}
(ROOT / 'calibration_inputs.json').write_text(
    json.dumps(calibration_records, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
(ROOT / 'calibration_overlap_audit.json').write_text(
    json.dumps(overlap_audit, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)

WARMUP_INDICES = [0, 825, 1650]
MEASURED_INDICES = [0, 34, 103, 171, 240, 309, 378, 447, 515, 584, 653, 722, 791,
                    825, 859, 928, 997, 1066, 1135, 1203, 1272, 1341, 1410, 1479,
                    1547, 1616, 1650]

STUDY_DESIGN = {
    'study_id': 'paddleocr_vl_1_6_l4_gptq_wna16_v1',
    'hardware': {'required_gpu_count': 4, 'required_gpu_name': 'NVIDIA L4', 'compute_capability': '8.9'},
    'variants': [
        {'id': 'BF16_W16A16', 'checkpoint': 'models/PaddleOCR-VL-1.6-BF16', 'weights': 'BF16', 'activations': 'BF16'},
        {'id': 'GPTQ_W8A16_G128', 'checkpoint': 'models/PaddleOCR-VL-1.6-GPTQ-W8A16-G128', 'weights': 'INT8', 'activations': 'BF16'},
        {'id': 'GPTQ_W4A16_G128', 'checkpoint': 'models/PaddleOCR-VL-1.6-GPTQ-W4A16-G128', 'weights': 'INT4', 'activations': 'BF16'},
    ],
    'quantization': {
        'offline': True, 'algorithm': 'GPTQ', 'group_size': 128,
        'num_calibration_samples': EXPECTED_CALIBRATION, 'calibration_seed': SEED,
        'calibration_dataset': 'pierreguillou/DocLayNet-base',
        'calibration_source_split': 'train',
        'calibration_source_pool_pages': 6910,
        'calibration_inputs': 'calibration_inputs.json',
        'calibration_overlap_audit': 'calibration_overlap_audit.json',
        'calibration_is_disjoint_from_test': True,
        'activation_dtype_for_both_low_bit_variants': 'BF16',
    },
    'logical_scope': {
        'vision_attention_layers': 27,
        'vision_logical_matrices_per_layer': ['q_proj', 'k_proj', 'v_proj', 'out_proj'],
        'vision_logical_count': 108,
        'decoder_layers': 18,
        'decoder_logical_matrices_per_layer': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        'decoder_logical_count': 126,
        'total_logical_matrices': 234,
        'expected_fused_runtime_linear_modules': 126,
    },
    'full_precision_exclusions': [
        'input token embedding', 'lm_head', 'vision MLP fc1/fc2',
        'mlp_AR multimodal projector', 'vision patch convolution / positional embedding',
        'all normalization layers', 'PP-DocLayoutV3',
    ],
}
RUNTIME_CONTRACT = {
    'benchmark_pages': EXPECTED_PAGES,
    'warmup': {'manifest_indices': WARMUP_INDICES},
    'measured': {'manifest_indices': MEASURED_INDICES},
    'generation': {
        'prompt': 'OCR:', 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.0,
        'max_new_tokens': 4096, 'min_pixels': 112896, 'max_pixels': 1003520,
        'skip_special_tokens': True, 'use_cache': True,
    },
    'vllm': {
        'version': VERSIONS['vllm'], 'dtype': 'bfloat16',
        'request_concurrency_values': [6, 12, 24], 'max_num_seqs': 24,
        'gpu_memory_utilization': 0.82, 'max_model_len': 8192,
        'max_num_batched_tokens': 16384, 'enable_prefix_caching': False,
        'quantized_linear_backend': 'marlin', 'native_kernel_validation_required': True,
    },
    'gpu_assignment': {'vllm_server_physical_gpu': 0, 'paddle_layout_physical_gpu': 1, 'unused_physical_gpus': [2, 3]},
    'failure_policy': {'retries_after_first_failure': 1, 'fail_variant_after_second_failure': True},
    'primary_timing': {'includes_server_startup': False, 'includes_warmup': False, 'includes_file_saving': False},
    'quality_output_concurrency': 24,
}

for filename, payload in [
    ('study_design.json', STUDY_DESIGN),
    ('benchmark_runtime_config.json', RUNTIME_CONTRACT),
]:
    (ROOT / filename).write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

assert len(benchmark_inputs) == EXPECTED_PAGES
assert len(MEASURED_INDICES) == 27 and len(set(MEASURED_INDICES)) == 27
assert json.loads((ROOT / 'benchmark_inputs.json').read_text(encoding='utf-8')) == benchmark_inputs
print('SOURCE + CONTRACT VALIDATION: PASS')
print('BF16 source tensor dtypes:', source_dtypes)
print('Test pages:', len(benchmark_inputs))
print('Measured pages:', MEASURED_INDICES)


## 4. Store the offline GPTQ worker

The worker discovers the model's actual `torch.nn.Linear` names, selects exactly the approved 108 vision-attention and 126 decoder matrices, and ignores every other linear. It writes the scope audit before quantization and refuses to continue unless the layer indices and count are exact.


In [ ]:
QUANT_WORKER_SOURCE = r'''
import argparse, hashlib, json, os, random, re
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor
from llmcompressor import oneshot
try:
    from llmcompressor.modifiers.quantization import GPTQModifier
except ImportError:
    from llmcompressor.modifiers.gptq import GPTQModifier

parser = argparse.ArgumentParser()
parser.add_argument('--source', required=True)
parser.add_argument('--output', required=True)
parser.add_argument('--bundle', required=True)
parser.add_argument('--calibration-manifest', required=True)
parser.add_argument('--scheme', choices=['W8A16', 'W4A16'], required=True)
parser.add_argument('--group-size', type=int, default=128)
parser.add_argument('--seed', type=int, default=2026)
args = parser.parse_args()

source = Path(args.source)
output = Path(args.output)
bundle = Path(args.bundle)
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
torch.set_grad_enabled(False)

records = json.loads(Path(args.calibration_manifest).read_text(encoding='utf-8'))
if len(records) != 512:
    raise RuntimeError(f'Expected 512 calibration inputs, got {len(records)}')
for record in records:
    image_path = bundle / record['image']
    digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
    if digest != record['sha256']:
        raise RuntimeError(f'Calibration image hash mismatch: {image_path}')

print('Loading BF16 source model...', flush=True)
model = AutoModelForImageTextToText.from_pretrained(
    str(source), trust_remote_code=True, torch_dtype=torch.bfloat16,
    device_map='auto', low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(str(source), trust_remote_code=True)
model.eval()

vision_pattern = re.compile(
    r'(?:^|\\.)vision_model\\.encoder\\.layers\\.(\\d+)\\.self_attn\\.(q_proj|k_proj|v_proj|out_proj)$'
)
decoder_pattern = re.compile(
    r'(?:^|\\.)language_model\\.model\\.layers\\.(\\d+)\\.(?:self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))$'
)
all_linears = []
targets = []
vision_layers = set()
decoder_layers = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        all_linears.append(name)
        vision_match = vision_pattern.search(name)
        decoder_match = decoder_pattern.search(name)
        if vision_match:
            targets.append(name)
            vision_layers.add(int(vision_match.group(1)))
        elif decoder_match:
            targets.append(name)
            decoder_layers.add(int(decoder_match.group(1)))

targets = sorted(targets)
all_linears = sorted(all_linears)
ignored = sorted(set(all_linears) - set(targets))
audit = {
    'scheme': args.scheme,
    'algorithm': 'GPTQ',
    'group_size': args.group_size,
    'activation_dtype': 'BF16',
    'calibration_seed': args.seed,
    'calibration_count': len(records),
    'all_linear_count': len(all_linears),
    'target_count': len(targets),
    'ignored_linear_count': len(ignored),
    'vision_layer_indices': sorted(vision_layers),
    'decoder_layer_indices': sorted(decoder_layers),
    'target_names': targets,
    'ignored_linear_names': ignored,
}
output.mkdir(parents=True, exist_ok=True)
(output / 'scope_audit.pre_quantization.json').write_text(
    json.dumps(audit, indent=2) + '\n', encoding='utf-8'
)
if len(targets) != 234:
    raise RuntimeError(f'Quantization scope drift: expected 234 targets, got {len(targets)}. See scope audit.')
if vision_layers != set(range(27)):
    raise RuntimeError(f'Vision layer drift: {sorted(vision_layers)}')
if decoder_layers != set(range(18)):
    raise RuntimeError(f'Decoder layer drift: {sorted(decoder_layers)}')

dataset = Dataset.from_list([{'image': record['image']} for record in records])

def data_collator(batch):
    if len(batch) != 1:
        raise RuntimeError(f'Calibration collator requires batch size 1, got {len(batch)}')
    image_path = bundle / batch[0]['image']
    with Image.open(image_path) as opened:
        image = opened.convert('RGB')
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': 'OCR:'},
        ],
    }]
    encoded = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
        images_kwargs={'min_pixels': 112896, 'max_pixels': 1003520},
    )
    return dict(encoded)

# The built-in W8A16 and W4A16 presets are static per-group schemes with
# group_size=128. The saved compressed-tensors config is audited below.
modifier = GPTQModifier(targets='Linear', scheme=args.scheme, ignore=ignored)
print(json.dumps({
    'scheme': args.scheme, 'targets': len(targets), 'ignored': len(ignored),
    'calibration_examples': len(records), 'group_size': args.group_size,
}, indent=2), flush=True)

oneshot(
    model=model,
    dataset=dataset,
    recipe=modifier,
    data_collator=data_collator,
    num_calibration_samples=len(records),
    max_seq_length=8192,
    sequential_targets=['Linear'],
    shuffle_calibration_samples=False,
)

model.save_pretrained(str(output), save_compressed=True, safe_serialization=True)
processor.save_pretrained(str(output))
(output / 'scope_audit.json').write_text(json.dumps(audit, indent=2) + '\n', encoding='utf-8')

config = json.loads((output / 'config.json').read_text(encoding='utf-8'))
quant_config = config.get('quantization_config')
if not isinstance(quant_config, dict):
    raise RuntimeError('Compressed checkpoint config has no quantization_config.')
print('QUANTIZATION COMPLETE:', args.scheme, output, flush=True)
'''

quant_worker = SCRIPTS / 'quantize_wna16.py'
quant_worker.write_text(QUANT_WORKER_SOURCE.lstrip(), encoding='utf-8')
compile(quant_worker.read_text(encoding='utf-8'), str(quant_worker), 'exec')
print('Quantization worker:', quant_worker)


## 5. Store the fixed PaddleOCR client benchmark worker

The offline test notebook launches this worker in the isolated Paddle environment. It preserves the original layout/VLM pipeline and uses a local HTTP proxy only to attribute vLLM calls and token counts to each page.


In [ ]:
CLIENT_WORKER_SOURCE = r'''
import json, os, random, statistics, threading, time, traceback, urllib.error, urllib.request
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import numpy as np
import paddle
from paddleocr import PaddleOCRVL

BUNDLE = Path(os.environ['BUNDLE'])
RESULT_DIR = Path(os.environ['RESULT_DIR'])
LAYOUT_DIR = Path(os.environ['LAYOUT_DIR'])
SERVER_URL = os.environ['SERVER_URL'].rstrip('/')
PROXY_PORT = int(os.environ['PROXY_PORT'])
SERVED_NAME = os.environ['SERVED_NAME']
VARIANT = os.environ['VARIANT']
CONCURRENCY = int(os.environ['REQUEST_CONCURRENCY'])
SAVE_OUTPUTS = os.environ.get('SAVE_OUTPUTS', '0') == '1'
runtime = json.loads(Path(os.environ['RUNTIME_CONTRACT']).read_text(encoding='utf-8'))
inputs = json.loads(Path(os.environ['BENCHMARK_INPUTS']).read_text(encoding='utf-8'))
warmup_indices = json.loads(os.environ['WARMUP_INDICES_JSON'])
measured_indices = json.loads(os.environ['MEASURED_INDICES_JSON'])

assert len(inputs) == runtime['benchmark_pages'] == 1651
assert warmup_indices == runtime['warmup']['manifest_indices']
assert measured_indices == runtime['measured']['manifest_indices']

RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / 'markdown').mkdir(parents=True, exist_ok=True)
(RESULT_DIR / 'pages').mkdir(parents=True, exist_ok=True)
metrics_path = RESULT_DIR / 'page_metrics.jsonl'
request_log_path = RESULT_DIR / 'vlm_request_log.jsonl'
random.seed(2026)
np.random.seed(2026)
paddle.disable_static()
paddle.set_device('gpu:0')
paddle.seed(2026)

predict_kwargs = {
    'min_pixels': int(runtime['generation']['min_pixels']),
    'max_pixels': int(runtime['generation']['max_pixels']),
    'max_new_tokens': int(runtime['generation']['max_new_tokens']),
    'repetition_penalty': 1.0, 'temperature': 0.0, 'top_p': 1.0,
}

def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + '\n', encoding='utf-8')
    temporary.replace(path)

def append_jsonl(path, payload):
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')
        stream.flush()
        os.fsync(stream.fileno())

proxy_state = {'page_id': None, 'manifest_index': None, 'phase': None, 'attempt_id': None}
proxy_lock = threading.Lock()
request_records = []
request_counter = 0
upstream_base = SERVER_URL.removesuffix('/v1')

class ProxyHandler(BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'
    def log_message(self, fmt, *args):
        return
    def do_GET(self):
        self.forward()
    def do_POST(self):
        self.forward()
    def forward(self):
        global request_counter
        body = self.rfile.read(int(self.headers.get('Content-Length', '0') or 0))
        with proxy_lock:
            request_counter += 1
            request_id = request_counter
            context = dict(proxy_state)
        started = time.perf_counter()
        status = 599
        response_body = b''
        error = None
        try:
            request = urllib.request.Request(
                upstream_base + self.path,
                data=body if self.command != 'GET' else None,
                headers={'Content-Type': self.headers.get('Content-Type', 'application/json')},
                method=self.command,
            )
            with urllib.request.urlopen(request, timeout=1800) as response:
                status = response.status
                response_body = response.read()
                content_type = response.headers.get('Content-Type', 'application/json')
        except urllib.error.HTTPError as exc:
            status = exc.code
            response_body = exc.read()
            content_type = 'application/json'
            error = repr(exc)
        except Exception as exc:
            content_type = 'application/json'
            error = repr(exc)
            response_body = json.dumps({'error': error}).encode()
        ended = time.perf_counter()
        usage = {}
        finish_reasons = []
        try:
            parsed = json.loads(response_body.decode('utf-8'))
            usage = parsed.get('usage') or {}
            finish_reasons = [choice.get('finish_reason') for choice in parsed.get('choices', []) if isinstance(choice, dict)]
        except Exception:
            pass
        record = {
            'request_id': request_id, **context, 'method': self.command, 'path': self.path,
            'status_code': status, 'started_perf': started, 'ended_perf': ended,
            'request_seconds': ended - started, 'usage': usage,
            'finish_reasons': finish_reasons, 'error': error,
        }
        with proxy_lock:
            request_records.append(record)
        append_jsonl(request_log_path, record)
        self.send_response(status)
        self.send_header('Content-Type', content_type)
        self.send_header('Content-Length', str(len(response_body)))
        self.end_headers()
        self.wfile.write(response_body)

proxy = ThreadingHTTPServer(('127.0.0.1', PROXY_PORT), ProxyHandler)
threading.Thread(target=proxy.serve_forever, daemon=True).start()
proxy_url = f'http://127.0.0.1:{PROXY_PORT}/v1'

def set_context(page_id=None, index=None, phase=None, attempt_id=None):
    with proxy_lock:
        proxy_state.update(page_id=page_id, manifest_index=index, phase=phase, attempt_id=attempt_id)

def summarize_requests(attempt_ids):
    with proxy_lock:
        records = [r for r in request_records if r.get('attempt_id') in attempt_ids and r['method'] == 'POST']
    if not records:
        return {'vlm_calls': 0, 'vlm_seconds': 0.0, 'vlm_request_seconds_sum': 0.0,
                'generated_tokens': 0, 'token_limit_reached': False}
    generated = sum(int((r.get('usage') or {}).get('completion_tokens') or 0) for r in records)
    starts = [r['started_perf'] for r in records]
    ends = [r['ended_perf'] for r in records]
    return {
        'vlm_calls': len(records), 'vlm_seconds': max(ends) - min(starts),
        'vlm_request_seconds_sum': sum(r['request_seconds'] for r in records),
        'generated_tokens': generated,
        'token_limit_reached': any(reason == 'length' for r in records for reason in r.get('finish_reasons', [])),
    }

def save_result(page_id, result):
    page_dir = RESULT_DIR / 'pages' / page_id
    page_dir.mkdir(parents=True, exist_ok=True)
    markdown_path = page_dir / f'{page_id}.md'
    json_path = page_dir / f'{page_id}.json'
    result.save_to_markdown(str(markdown_path), pretty=False, show_formula_number=False)
    result.save_to_json(str(json_path))
    text = markdown_path.read_text(encoding='utf-8')
    for old, new in {
        '(imgs/': f'(../pages/{page_id}/imgs/', '(./imgs/': f'(../pages/{page_id}/imgs/',
        'src="imgs/': f'src="../pages/{page_id}/imgs/', 'src="./imgs/': f'src="../pages/{page_id}/imgs/',
    }.items():
        text = text.replace(old, new)
    (RESULT_DIR / 'markdown' / f'{page_id}.md').write_text(text, encoding='utf-8')
    json.loads(json_path.read_text(encoding='utf-8'))
    return text

def run_page(index, phase, save_outputs):
    entry = inputs[index]
    page_id = entry['page_id']
    image_path = BUNDLE / entry['image']
    attempt_ids = []
    result = None
    error = None
    elapsed = 0.0
    for attempt in range(runtime['failure_policy']['retries_after_first_failure'] + 1):
        attempt_id = f'{phase}-{page_id}-{attempt}-{time.time_ns()}'
        attempt_ids.append(attempt_id)
        set_context(page_id, index, phase, attempt_id)
        started = time.perf_counter()
        try:
            with paddle.no_grad():
                predictions = list(pipeline.predict(str(image_path), **predict_kwargs))
            paddle.device.synchronize()
            elapsed += time.perf_counter() - started
            if len(predictions) != 1:
                raise RuntimeError(f'Expected one result, got {len(predictions)}')
            result = predictions[0]
            error = None
            break
        except Exception:
            paddle.device.synchronize()
            elapsed += time.perf_counter() - started
            error = traceback.format_exc()
            print(error, flush=True)
        finally:
            set_context()
    request_summary = summarize_requests(set(attempt_ids))
    row = {
        'variant': VARIANT, 'request_concurrency': CONCURRENCY, 'phase': phase,
        'manifest_index': index, 'page_id': page_id, 'page_inference_seconds': elapsed,
        **request_summary, 'attempts': len(attempt_ids),
        'status': 'completed' if result is not None else 'failed', 'error_or_timeout': error,
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    }
    if result is not None:
        try:
            row['layout_blocks'] = len(result['parsing_res_list'])
        except Exception:
            row['layout_blocks'] = None
        if save_outputs:
            text = save_result(page_id, result)
            row['markdown_characters'] = len(text)
            row['markdown_nonempty'] = bool(text.strip())
    return row

try:
    startup = time.perf_counter()
    pipeline = PaddleOCRVL(
        pipeline_version='v1.6', device='gpu:0',
        layout_detection_model_name='PP-DocLayoutV3', layout_detection_model_dir=str(LAYOUT_DIR),
        vl_rec_backend='vllm-server', vl_rec_server_url=proxy_url,
        vl_rec_api_model_name=SERVED_NAME, vl_rec_max_concurrency=CONCURRENCY,
        use_doc_orientation_classify=False, use_doc_unwarping=False, use_layout_detection=True,
        use_chart_recognition=False, use_seal_recognition=False, use_ocr_for_image_block=False,
        use_queues=True, format_block_content=False, merge_layout_blocks=True,
        markdown_ignore_labels=[], enable_hpi=False, use_tensorrt=False, enable_cinn=False,
    )
    paddle.device.synchronize()
    startup_seconds = time.perf_counter() - startup
    atomic_json(RESULT_DIR / 'run_manifest.json', {
        'variant': VARIANT, 'request_concurrency': CONCURRENCY,
        'client_startup_seconds': startup_seconds, 'predict_kwargs': predict_kwargs,
        'save_outputs': SAVE_OUTPUTS, 'pipeline_fixed': True,
    })

    warmup_rows = []
    for index in warmup_indices:
        row = run_page(index, 'warmup', False)
        warmup_rows.append(row)
        print('warmup', VARIANT, CONCURRENCY, index, row['status'], f"{row['page_inference_seconds']:.2f}s", flush=True)
        if row['status'] != 'completed':
            raise RuntimeError(f'Warmup failed at index {index}')

    measured_rows = []
    print('MEASUREMENT_START', flush=True)
    for index in measured_indices:
        row = run_page(index, 'measured', SAVE_OUTPUTS)
        measured_rows.append(row)
        append_jsonl(metrics_path, row)
        print('measured', VARIANT, CONCURRENCY, index, row['status'], f"{row['page_inference_seconds']:.2f}s", flush=True)
        if row['status'] != 'completed':
            raise RuntimeError(f'Measured page failed at index {index}')
    print('MEASUREMENT_END', flush=True)

    total_seconds = sum(row['page_inference_seconds'] for row in measured_rows)
    total_vlm_seconds = sum(row['vlm_seconds'] for row in measured_rows)
    total_request_seconds = sum(row['vlm_request_seconds_sum'] for row in measured_rows)
    total_tokens = sum(row['generated_tokens'] for row in measured_rows)
    total_characters = sum(int(row.get('markdown_characters') or 0) for row in measured_rows) if SAVE_OUTPUTS else None
    summary = {
        'variant': VARIANT, 'request_concurrency': CONCURRENCY, 'status': 'PASS',
        'page_count': len(measured_rows), 'warmup_pages': warmup_rows,
        'total_page_inference_seconds': total_seconds, 'pages_per_hour': len(measured_rows) * 3600 / total_seconds,
        'total_vlm_seconds': total_vlm_seconds, 'total_vlm_request_seconds_sum': total_request_seconds,
        'total_generated_tokens': total_tokens,
        'tokens_per_vlm_second': total_tokens / total_vlm_seconds if total_vlm_seconds else None,
        'total_markdown_characters': total_characters,
        'saved_markdown_files': len(list((RESULT_DIR / 'markdown').glob('*.md'))),
        'measured_pages': measured_rows,
    }
    atomic_json(RESULT_DIR / 'speed_summary.json', summary)
    print(json.dumps({k: summary[k] for k in [
        'variant', 'request_concurrency', 'total_page_inference_seconds',
        'pages_per_hour', 'tokens_per_vlm_second', 'saved_markdown_files'
    ]}, indent=2), flush=True)
finally:
    proxy.shutdown()
    proxy.server_close()
'''

client_worker = SCRIPTS / 'l4_wna16_client_worker.py'
client_worker.write_text(CLIENT_WORKER_SOURCE.lstrip(), encoding='utf-8')
compile(client_worker.read_text(encoding='utf-8'), str(client_worker), 'exec')
print('Client benchmark worker:', client_worker)


## 6. Final bundle audit

This is the only cell whose success should be used as the signal to save and attach the dependency notebook output.


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


calibration_inputs = json.loads(
    (ROOT / 'calibration_inputs.json').read_text(encoding='utf-8')
)
separation_audit = json.loads(
    (ROOT / 'calibration_overlap_audit.json').read_text(encoding='utf-8')
)

assert len(calibration_inputs) == EXPECTED_CALIBRATION == 512
assert separation_audit['status'] == 'PASS'
assert separation_audit['calibration_images'] == 512
assert separation_audit['calibration_unique_source_documents'] == 512
assert separation_audit['omnidocbench_images_compared'] == 1651
assert (
    separation_audit['selected_per_category']
    == separation_audit['category_quota']
)
assert len(
    json.loads(
        (ROOT / 'benchmark_inputs.json').read_text(encoding='utf-8')
    )
) == 1651

verify_wheels(
    CLIENT_WHEELS,
    (
        'paddlepaddle_gpu-3.2.1',
        'paddleocr-3.7.0',
        'paddlex-3.7.2',
    ),
)
verify_wheels(
    SERVER_WHEELS,
    (
        'vllm-0.29.0',
        'compressed_tensors-0.17.0',
    ),
)
verify_wheels(
    QUANT_WHEELS,
    (
        'llmcompressor-0.12.0.1',
        'compressed_tensors-0.17.1',
    ),
)

require_file(BF16 / 'config.json')

# PP-DocLayoutV3 is a Paddle inference model and does not use config.json.
for filename in (
    'inference.json',
    'inference.pdiparams',
    'inference.yml',
):
    require_file(LAYOUT / filename)

require_file(SCRIPTS / 'quantize_wna16.py')
require_file(SCRIPTS / 'l4_wna16_client_worker.py')

wheel_manifest = {}
for group, directory in [
    ('paddle_client', CLIENT_WHEELS),
    ('vllm_server', SERVER_WHEELS),
    ('llmcompressor_quant', QUANT_WHEELS),
]:
    wheel_manifest[group] = [
        {
            'filename': path.name,
            'bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        }
        for path in sorted(directory.glob('*.whl'))
    ]

manifest = {
    'bundle': 'paddleocr_vl_1_6_l4_wna16_bundle_v1',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'versions': VERSIONS,
    'source_revisions': REVISIONS,
    'paths': {
        'client_wheels': CLIENT_WHEELS.relative_to(ROOT).as_posix(),
        'server_wheels': SERVER_WHEELS.relative_to(ROOT).as_posix(),
        'quant_wheels': QUANT_WHEELS.relative_to(ROOT).as_posix(),
        'bf16': BF16.relative_to(ROOT).as_posix(),
        'layout': LAYOUT.relative_to(ROOT).as_posix(),
        'omnidocbench': OMNI.relative_to(ROOT).as_posix(),
        'benchmark_inputs': 'benchmark_inputs.json',
        'calibration_inputs': 'calibration_inputs.json',
        'calibration_overlap_audit': 'calibration_overlap_audit.json',
        'omnidocbench_image_fingerprints':
            'omnidocbench_image_fingerprints.json',
        'study_design': 'study_design.json',
        'runtime_contract': 'benchmark_runtime_config.json',
        'quant_worker': 'scripts/quantize_wna16.py',
        'client_worker': 'scripts/l4_wna16_client_worker.py',
    },
    'quantized_output_names': {
        'W8A16': 'PaddleOCR-VL-1.6-GPTQ-W8A16-G128',
        'W4A16': 'PaddleOCR-VL-1.6-GPTQ-W4A16-G128',
    },
    'matched_low_bit_contract': {
        'schemes': ['W8A16', 'W4A16'],
        'algorithm': 'GPTQ',
        'group_size': 128,
        'activation_dtype': 'BF16',
        'calibration_examples': 512,
        'logical_target_count': 234,
        'target_name_lists_identical':
            'required_and_audited_after_quantization',
    },
    'wheelhouse': wheel_manifest,
}

(ROOT / 'bundle_manifest.json').write_text(
    json.dumps(manifest, indent=2) + '\n',
    encoding='utf-8',
)

# Reparse every JSON contract.
for path in [
    ROOT / 'bundle_manifest.json',
    ROOT / 'benchmark_inputs.json',
    ROOT / 'calibration_inputs.json',
    ROOT / 'calibration_overlap_audit.json',
    ROOT / 'omnidocbench_image_fingerprints.json',
    ROOT / 'study_design.json',
    ROOT / 'benchmark_runtime_config.json',
    ROOT / 'source_revisions.json',
]:
    json.loads(path.read_text(encoding='utf-8'))

shutil.rmtree(CALIBRATION_DOWNLOAD, ignore_errors=True)
shutil.rmtree(TOOLS_SITE, ignore_errors=True)

total_gib = sum(
    path.stat().st_size
    for path in ROOT.rglob('*')
    if path.is_file()
) / 2**30

print('=' * 88)
print('INTERNET-ON DEPENDENCY BUNDLE: PASS')
print('Bundle:', ROOT)
print('BF16 source:', BF16.name)
print('Offline quantization worker:', SCRIPTS / 'quantize_wna16.py')
print(
    'Calibration examples:',
    EXPECTED_CALIBRATION,
    '(disjoint from OmniDocBench)',
)
print('OmniDocBench pages:', EXPECTED_PAGES)
print(f'Bundle size: {total_gib:.3f} GiB')
print(
    'Save this version, attach its output to notebook 2, '
    'select 4×L4, and keep Internet OFF there.'
)
print('=' * 88)